In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os

In [3]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to C:\Users\saniya G
[nltk_data]     Madhavi\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [4]:
# Step 1: Load the Dataset
apps_df = pd.read_csv(os.path.join("googleplaystore.csv", "googleplaystore.csv"))
reviews_df = pd.read_csv(os.path.join("googleplaystore_user_reviews.csv", "googleplaystore_user_reviews.csv"))

In [5]:
# Step 2: Data Cleaning
apps_df = apps_df.dropna(subset=['Rating'])
for column in apps_df.columns:
    apps_df[column].fillna(apps_df[column].mode()[0], inplace=True)
apps_df.drop_duplicates(inplace=True)
apps_df = apps_df[apps_df['Rating'] <= 5]
reviews_df.dropna(subset=['Translated_Review'], inplace=True)

C:\Users\saniya G Madhavi\AppData\Local\Temp\ipykernel_10404\3675673569.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  apps_df[column].fillna(apps_df[column].mode()[0], inplace=True)


In [6]:
# Merge datasets on 'App' and handle non-matching apps
merged_df = pd.merge(apps_df, reviews_df, on='App', how='inner')

In [7]:
# Step 3: Data Transformation
apps_df['Reviews'] = apps_df['Reviews'].astype(int)
apps_df['Installs'] = apps_df['Installs'].str.replace(',', '').str.replace('+', '').astype(int)
apps_df['Price'] = apps_df['Price'].str.replace('$', '').astype(float)

In [8]:
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M', ''))
    elif 'k' in size:
        return float(size.replace('k', '')) / 1024
    else:
        return np.nan

In [9]:
apps_df['Size'] = apps_df['Size'].apply(convert_size)

In [10]:
# Add log_installs and log_reviews columns
apps_df['Log_Installs'] = np.log1p(apps_df['Installs'])
apps_df['Log_Reviews'] = np.log1p(apps_df['Reviews'])

In [11]:
# Add Rating Group column
def rating_group(rating):
    if rating >= 4:
        return 'Top rated'
    elif rating >= 3:
        return 'Above average'
    elif rating >= 2:
        return 'Average'
    else:
        return 'Below average'

apps_df['Rating_Group'] = apps_df['Rating'].apply(rating_group)

In [12]:
# Add Revenue column
apps_df['Revenue'] = apps_df['Price'] * apps_df['Installs']

In [13]:
# Sentiment Analysis
sia = SentimentIntensityAnalyzer()
reviews_df['Sentiment_Score'] = reviews_df['Translated_Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

In [14]:
# Extract year from 'Last Updated' and create 'Year' column
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')
apps_df['Year'] = apps_df['Last Updated'].dt.year

In [15]:
import plotly.express as px
import plotly.graph_objects as go

# Define the path for your HTML files
html_files_path = "./"

# Make sure the directory exists
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

# Initialize plot_containers
plot_containers = ""

# Save each Plotly figure to an HTML file
def save_plot_as_html(fig, filename, insight):
    global plot_containers
    filepath = os.path.join(html_files_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    # Append the plot and its insight to plot_containers
    plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    fig.write_html(filepath, full_html=False, include_plotlyjs='inline')

# Define your plots
plot_width = 400
plot_height = 300
plot_bg_color = 'black'
text_color = 'white'
title_font = {'size': 16}
axis_font = {'size': 12}

# Category Analysis Plot
category_counts = apps_df['Category'].value_counts().nlargest(10)
fig1 = px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={'x': 'Category', 'y': 'Count'},
    title='Top Categories on Play Store',
    color=category_counts.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=plot_width,
    height=plot_height
)
fig1.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig1.update_traces(marker=dict(line=dict(color=text_color, width=1)))
save_plot_as_html(fig1, "category_analysis.html", "The top categories on the Play Store are dominated by tools, entertainment, and productivity apps. This suggests users are looking for apps that either provide utility or offer leisure activities.")

# Type Analysis Plot
type_counts = apps_df['Type'].value_counts()
fig2 = px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='App Type Distribution',
    color_discrete_sequence=px.colors.sequential.RdBu,
    width=plot_width,
    height=plot_height
)
fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig2, "type_analysis.html", "Most apps on the Play Store are free, indicating a strategy to attract users first and monetize through ads or in-app purchases.")

# Rating Distribution Plot
fig3 = px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#636EFA'],
    width=plot_width,
    height=plot_height
)
fig3.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig3, "rating_distribution.html", "Ratings are skewed towards higher values, suggesting that most apps are rated favorably by users.")

sentiment_counts = reviews_df['Sentiment_Score'].value_counts()
fig4 = px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    labels={'x': 'Sentiment Score', 'y': 'Count'},
    title='Sentiment Distribution',
    color=sentiment_counts.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=plot_width,
    height=plot_height
)
fig4.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig4.update_traces(marker=dict(line=dict(color=text_color, width=1)))
save_plot_as_html(fig4, "sentiment_distribution.html", "Sentiments in reviews show a mix of positive and negative feedback, with a slight lean towards positive sentiments.")

# Installs by Category Plot
installs_by_category = apps_df.groupby('Category')['Installs'].sum().nlargest(10)
fig5 = px.bar(
    x=installs_by_category.values,
    y=installs_by_category.index,
    orientation='h',
    labels={'x': 'Installs', 'y': 'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Blues,
    width=plot_width,
    height=plot_height
)
fig5.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig5.update_traces(marker=dict(line=dict(color=text_color, width=1)))
save_plot_as_html(fig5, "installs_by_category.html", "The categories with the most installs are social and communication apps, which reflects their broad appeal and daily usage.")

# Updates Per Year Plot
updates_per_year = apps_df['Last Updated'].dt.year.value_counts().sort_index()
fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x': 'Year', 'y': 'Number of Updates'},
    title='Number of Updates Over the Years',
    color_discrete_sequence=['#AB63FA'],
    width=plot_width,
    height=plot_height
)
fig6.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig6, "updates_per_year.html", "Updates have been increasing over the years, showing that developers are actively maintaining and improving their apps.")

# Revenue by Category Plot
revenue_by_category = apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7 = px.bar(
    x=revenue_by_category.index,
    y=revenue_by_category.values,
    labels={'x': 'Category', 'y': 'Revenue'},
    title='Revenue by Category',
    color=revenue_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=plot_width,
    height=plot_height
)
fig7.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig7.update_traces(marker=dict(line=dict(color=text_color, width=1)))
save_plot_as_html(fig7, "revenue_by_category.html", "Categories such as Business and Productivity lead in revenue generation, indicating their monetization potential.")

# Genre Count Plot
genre_counts = apps_df['Genres'].str.split(';', expand=True).stack().value_counts().nlargest(10)
fig8 = px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x': 'Genre', 'y': 'Count'},
    title='Top Genres',
    color=genre_counts.index,
    color_discrete_sequence=px.colors.sequential.OrRd,
    width=plot_width,
    height=plot_height
)
fig8.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig8.update_traces(marker=dict(line=dict(color=text_color, width=1)))
save_plot_as_html(fig8, "genres_counts.html", "Action and Casual genres are the most common, reflecting users' preference for engaging and easy-to-play games.")

# Impact of Last Update on Rating
fig9 = px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=plot_width,
    height=plot_height
)
fig9.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig9, "update_on_rating.html", "The scatter plot shows a weak correlation between the last update date and ratings, suggesting that more frequent updates don't always result in better ratings.")

# Ratings for Paid vs Free Apps
fig10 = px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Ratings for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=plot_width,
    height=plot_height
)
fig10.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig10, "ratings_paid_free.html", "Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for.")

# Task 1: Grouped bar chart visible only from 3 PM to 5 PM IST
jan_apps = apps_df[apps_df['Last Updated'].dt.month == 1]
task1_categories = (
    jan_apps.groupby('Category', as_index=False)
    .agg(
        Installs=('Installs', 'sum'),
        Average_Rating=('Rating', 'mean'),
        Average_Size=('Size', 'mean'),
        Total_Reviews=('Reviews', 'sum')
    )
)
task1_categories = (
    task1_categories[
        (task1_categories['Average_Rating'] >= 4.0) &
        (task1_categories['Average_Size'] >= 10)
    ]
    .sort_values('Installs', ascending=False)
    .head(10)
)

if not task1_categories.empty:
    fig11 = go.Figure()
    fig11.add_bar(
        x=task1_categories['Category'],
        y=task1_categories['Average_Rating'],
        name='Average Rating',
        marker_color='#00CC96'
    )
    fig11.add_bar(
        x=task1_categories['Category'],
        y=task1_categories['Total_Reviews'],
        name='Total Reviews',
        marker_color='#EF553B',
        yaxis='y2'
    )
    fig11.update_layout(
        title='Average Rating and Reviews for Top Installed Categories',
        barmode='group',
        width=plot_width,
        height=plot_height,
        plot_bgcolor=plot_bg_color,
        paper_bgcolor=plot_bg_color,
        font_color=text_color,
        title_font=title_font,
        xaxis=dict(title_font=axis_font, tickangle=-35),
        yaxis=dict(title='Average Rating', title_font=axis_font, range=[0, 5]),
        yaxis2=dict(title='Total Reviews', overlaying='y', side='right', showgrid=False),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=10, r=10, t=55, b=55)
    )
    fig11.update_traces(marker=dict(line=dict(color=text_color, width=1)))
    save_plot_as_html(
        fig11,
        "task1_rating_reviews_by_category.html",
        "This chart compares average rating and total reviews for the top 10 January-updated categories by installs after keeping categories with average rating at least 4.0 and average size at least 10 MB."
    )
# Task 2: Choropleth map visible only from 6 PM to 8 PM IST
task2_categories = (
    apps_df[~apps_df['Category'].str.upper().str.startswith(('A', 'C', 'G', 'S'), na=False)]
    .groupby('Category', as_index=False)['Installs']
    .sum()
    .sort_values('Installs', ascending=False)
    .head(5)
)
task2_locations = ['USA', 'IND', 'BRA', 'IDN', 'RUS']
task2_lat_lon = {
    'USA': (37.0902, -95.7129),
    'IND': (20.5937, 78.9629),
    'BRA': (-14.2350, -51.9253),
    'IDN': (-0.7893, 113.9213),
    'RUS': (61.5240, 105.3188)
}
task2_categories = task2_categories.assign(
    ISO_Code=task2_locations[:len(task2_categories)],
    Highlight=np.where(task2_categories['Installs'] > 1000000, 'Above 1M installs', '1M installs or below')
)
task2_categories['Latitude'] = task2_categories['ISO_Code'].map(lambda code: task2_lat_lon[code][0])
task2_categories['Longitude'] = task2_categories['ISO_Code'].map(lambda code: task2_lat_lon[code][1])

if not task2_categories.empty:
    fig12 = px.choropleth(
        task2_categories,
        locations='ISO_Code',
        color='Installs',
        hover_name='Category',
        hover_data={'ISO_Code': False, 'Installs': ':,', 'Highlight': True},
        color_continuous_scale='Viridis',
        title='Global Installs by Top App Categories',
        width=plot_width,
        height=plot_height
    )
    task2_highlight = task2_categories[task2_categories['Installs'] > 1000000]
    if not task2_highlight.empty:
        fig12.add_scattergeo(
            lat=task2_highlight['Latitude'],
            lon=task2_highlight['Longitude'],
            text=task2_highlight['Category'],
            mode='markers+text',
            marker=dict(size=18, color='rgba(255, 77, 109, 0.85)', line=dict(width=3, color='white')),
            textposition='top center',
            name='Installs > 1M',
            hovertemplate='%{text}<br>Installs exceed 1M<extra></extra>'
        )
    fig12.update_geos(bgcolor=plot_bg_color, showframe=False, showcoastlines=True)
    fig12.update_layout(
        plot_bgcolor=plot_bg_color,
        paper_bgcolor=plot_bg_color,
        font_color=text_color,
        title_font=title_font,
        margin=dict(l=10, r=10, t=45, b=10),
        coloraxis_colorbar=dict(title='Installs')
    )
    save_plot_as_html(
        fig12,
        "task2_global_installs_choropleth.html",
        "This map shows the top 5 eligible app categories by global installs, excluding categories starting with A, C, G, or S and visually highlighting categories above 1 million installs."
    )

# Task 3: Dual-axis chart visible only from 1 PM to 2 PM IST
def android_major_version(version):
    match = pd.Series([str(version)]).str.extract(r'(\d+(?:\.\d+)?)')[0].iloc[0]
    return float(match) if pd.notna(match) else np.nan

task3_apps = apps_df.copy()
task3_apps['Android_Version_Number'] = task3_apps['Android Ver'].apply(android_major_version)
task3_apps = task3_apps[
    (task3_apps['Installs'] >= 10000) &
    (task3_apps['Revenue'] >= 10000) &
    (task3_apps['Android_Version_Number'] > 4.0) &
    (task3_apps['Size'] > 15) &
    (task3_apps['Content Rating'] == 'Everyone') &
    (task3_apps['App'].str.len() <= 30)
]
task3_top_categories = task3_apps.groupby('Category')['Installs'].sum().nlargest(3).index
task3_summary = (
    task3_apps[task3_apps['Category'].isin(task3_top_categories)]
    .groupby(['Category', 'Type'], as_index=False)
    .agg(Average_Installs=('Installs', 'mean'), Revenue=('Revenue', 'sum'))
)
task3_summary['Category_Type'] = task3_summary['Category'] + ' - ' + task3_summary['Type']

if not task3_summary.empty:
    fig13 = go.Figure()
    fig13.add_bar(
        x=task3_summary['Category_Type'],
        y=task3_summary['Average_Installs'],
        name='Average Installs',
        marker_color='#636EFA'
    )
    fig13.add_scatter(
        x=task3_summary['Category_Type'],
        y=task3_summary['Revenue'],
        name='Revenue',
        mode='lines+markers',
        marker_color='#FFA15A',
        yaxis='y2'
    )
    fig13.update_layout(
        title='Average Installs and Revenue: Free vs Paid Apps',
        width=plot_width,
        height=plot_height,
        plot_bgcolor=plot_bg_color,
        paper_bgcolor=plot_bg_color,
        font_color=text_color,
        title_font=title_font,
        xaxis=dict(title_font=axis_font, tickangle=-35),
        yaxis=dict(title='Average Installs', title_font=axis_font),
        yaxis2=dict(title='Revenue', overlaying='y', side='right', showgrid=False),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=10, r=10, t=55, b=65)
    )
    save_plot_as_html(
        fig13,
        "task3_installs_revenue_free_paid.html",
        "This chart compares average installs and revenue for free vs paid apps in the top 3 filtered categories after applying install, revenue, Android version, size, content rating, and app-name length filters."
    )

# Task 4: Time series chart visible only from 6 PM to 9 PM IST
task4_apps = apps_df[
    apps_df['Category'].str.upper().str.startswith(('E', 'C', 'B'), na=False) &
    ~apps_df['App'].str.lower().str.startswith(('x', 'y', 'z'), na=False) &
    ~apps_df['App'].str.contains('s', case=False, na=False) &
    (apps_df['Reviews'] > 500)
].copy()
task4_apps['Update_Month'] = task4_apps['Last Updated'].dt.to_period('M').dt.to_timestamp()
task4_translations = {
    'BEAUTY': '\u0938\u094c\u0902\u0926\u0930\u094d\u092f',
    'BUSINESS': '\u0bb5\u0ba3\u0bbf\u0b95\u0bae\u0bcd',
    'DATING': 'Partnersuche'
}
task4_apps['Category_Key'] = task4_apps['Category'].str.upper()
task4_apps['Category_Display'] = task4_apps['Category_Key'].replace(task4_translations)
where_task4_untranslated = task4_apps['Category_Display'] == task4_apps['Category_Key']
task4_apps.loc[where_task4_untranslated, 'Category_Display'] = task4_apps.loc[where_task4_untranslated, 'Category']
task4_monthly = (
    task4_apps.dropna(subset=['Update_Month'])
    .groupby(['Update_Month', 'Category_Display'], as_index=False)['Installs']
    .sum()
    .sort_values(['Category_Display', 'Update_Month'])
)
task4_monthly['MoM_Growth'] = task4_monthly.groupby('Category_Display')['Installs'].pct_change()
task4_growth = task4_monthly[task4_monthly['MoM_Growth'] > 0.20]

if not task4_monthly.empty:
    fig14 = px.line(
        task4_monthly,
        x='Update_Month',
        y='Installs',
        color='Category_Display',
        markers=True,
        title='Monthly Install Trend by Category',
        labels={'Update_Month': 'Month', 'Installs': 'Total Installs', 'Category_Display': 'Category'},
        width=plot_width,
        height=plot_height
    )
    for category in task4_growth['Category_Display'].unique():
        growth_points = task4_growth[task4_growth['Category_Display'] == category]
        for _, row in growth_points.iterrows():
            fig14.add_scatter(
                x=[row['Update_Month'], row['Update_Month']],
                y=[0, row['Installs']],
                fill='tozeroy',
                mode='none',
                showlegend=False,
                hoverinfo='skip',
                fillcolor='rgba(255, 161, 90, 0.25)'
            )
    fig14.update_layout(
        plot_bgcolor=plot_bg_color,
        paper_bgcolor=plot_bg_color,
        font_color=text_color,
        title_font=title_font,
        xaxis=dict(title_font=axis_font),
        yaxis=dict(title_font=axis_font),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=10, r=10, t=55, b=45)
    )
    save_plot_as_html(
        fig14,
        "task4_installs_time_series.html",
        "This line chart tracks monthly installs by selected categories and shades months where installs grew by more than 20% month over month after applying the requested app, category, and review filters."
    )

# Task 5: Bubble chart visible only from 5 PM to 7 PM IST
task5_subjectivity = (
    reviews_df.groupby('App', as_index=False)['Sentiment_Subjectivity']
    .mean()
    .rename(columns={'Sentiment_Subjectivity': 'Average_Subjectivity'})
)
task5_categories = [
    'GAME', 'BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION',
    'DATING', 'ENTERTAINMENT', 'SOCIAL', 'EVENTS'
]
task5_apps = apps_df.merge(task5_subjectivity, on='App', how='inner')
task5_apps['Category_Key'] = task5_apps['Category'].str.upper()
task5_apps = task5_apps[
    (task5_apps['Rating'] > 3.5) &
    (task5_apps['Category_Key'].isin(task5_categories)) &
    (task5_apps['Reviews'] > 500) &
    ~task5_apps['App'].str.contains('s', case=False, na=False) &
    (task5_apps['Average_Subjectivity'] > 0.5) &
    (task5_apps['Installs'] > 50000)
].copy()
task5_translations = {
    'BEAUTY': '\u0938\u094c\u0902\u0926\u0930\u094d\u092f',
    'BUSINESS': '\u0bb5\u0ba3\u0bbf\u0b95\u0bae\u0bcd',
    'DATING': 'Partnersuche'
}
task5_apps['Category_Display'] = task5_apps['Category_Key'].replace(task5_translations)
where_task5_untranslated = task5_apps['Category_Display'] == task5_apps['Category_Key']
task5_apps.loc[where_task5_untranslated, 'Category_Display'] = task5_apps.loc[where_task5_untranslated, 'Category']
task5_game_label = task5_apps.loc[task5_apps['Category_Key'] == 'GAME', 'Category_Display'].drop_duplicates()
task5_color_map = {label: '#FF69B4' for label in task5_game_label}

if not task5_apps.empty:
    fig15 = px.scatter(
        task5_apps,
        x='Size',
        y='Rating',
        size='Installs',
        color='Category_Display',
        hover_name='App',
        hover_data={'Installs': ':,', 'Reviews': ':,', 'Average_Subjectivity': ':.2f', 'Size': ':.2f'},
        title='App Size vs Rating by Installs',
        labels={'Size': 'Size (MB)', 'Rating': 'Average Rating', 'Category_Display': 'Category'},
        color_discrete_map=task5_color_map,
        size_max=35,
        width=plot_width,
        height=plot_height
    )
    fig15.update_layout(
        plot_bgcolor=plot_bg_color,
        paper_bgcolor=plot_bg_color,
        font_color=text_color,
        title_font=title_font,
        xaxis=dict(title_font=axis_font),
        yaxis=dict(title_font=axis_font),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=10, r=10, t=55, b=45)
    )
    save_plot_as_html(
        fig15,
        "task5_size_rating_bubble.html",
        "This bubble chart compares app size and rating with bubble size based on installs after applying category, review, subjectivity, install, and app-name filters. The Game category is highlighted in pink."
    )

# Task 6: Stacked area chart visible only from 4 PM to 6 PM IST
task6_apps = apps_df[
    (apps_df['Rating'] >= 4.2) &
    ~apps_df['App'].str.contains(r'\d', na=False) &
    apps_df['Category'].str.upper().str.startswith(('T', 'P'), na=False) &
    (apps_df['Reviews'] > 1000) &
    (apps_df['Size'].between(20, 80))
].copy()
task6_apps['Update_Month'] = task6_apps['Last Updated'].dt.to_period('M').dt.to_timestamp()
task6_translations = {
    'TRAVEL_AND_LOCAL': 'Voyage et local',
    'PRODUCTIVITY': 'Productividad',
    'PHOTOGRAPHY': '\u5199\u771f'
}
task6_apps['Category_Key'] = task6_apps['Category'].str.upper()
task6_apps['Category_Display'] = task6_apps['Category_Key'].replace(task6_translations)
where_task6_untranslated = task6_apps['Category_Display'] == task6_apps['Category_Key']
task6_apps.loc[where_task6_untranslated, 'Category_Display'] = task6_apps.loc[where_task6_untranslated, 'Category']
task6_monthly = (
    task6_apps.dropna(subset=['Update_Month'])
    .groupby(['Update_Month', 'Category_Display'], as_index=False)['Installs']
    .sum()
    .sort_values(['Category_Display', 'Update_Month'])
)
task6_monthly['Cumulative_Installs'] = task6_monthly.groupby('Category_Display')['Installs'].cumsum()
task6_monthly['MoM_Growth'] = task6_monthly.groupby('Category_Display')['Installs'].pct_change()
task6_growth = task6_monthly[task6_monthly['MoM_Growth'] > 0.25]

if not task6_monthly.empty:
    task6_color_sequence = px.colors.qualitative.Bold
    task6_categories = task6_monthly['Category_Display'].drop_duplicates().tolist()
    task6_color_map = {
        category: task6_color_sequence[index % len(task6_color_sequence)]
        for index, category in enumerate(task6_categories)
    }
    fig16 = px.area(
        task6_monthly,
        x='Update_Month',
        y='Cumulative_Installs',
        color='Category_Display',
        title='Cumulative Installs Over Time by Category',
        labels={'Update_Month': 'Month', 'Cumulative_Installs': 'Cumulative Installs', 'Category_Display': 'Category'},
        color_discrete_map=task6_color_map,
        width=plot_width,
        height=plot_height
    )
    for category in task6_growth['Category_Display'].unique():
        category_months = task6_monthly[task6_monthly['Category_Display'] == category]
        growth_points = task6_growth[task6_growth['Category_Display'] == category]
        for _, row in growth_points.iterrows():
            previous_rows = category_months[category_months['Update_Month'] < row['Update_Month']]
            start_month = previous_rows['Update_Month'].max() if not previous_rows.empty else row['Update_Month']
            fig16.add_scatter(
                x=[start_month, row['Update_Month'], row['Update_Month'], start_month],
                y=[0, 0, row['Cumulative_Installs'], row['Cumulative_Installs']],
                fill='toself',
                mode='none',
                showlegend=False,
                hoverinfo='skip',
                fillcolor=task6_color_map.get(category, '#FFFFFF'),
                opacity=0.45
            )
    fig16.update_layout(
        plot_bgcolor=plot_bg_color,
        paper_bgcolor=plot_bg_color,
        font_color=text_color,
        title_font=title_font,
        xaxis=dict(title_font=axis_font),
        yaxis=dict(title_font=axis_font),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        margin=dict(l=10, r=10, t=55, b=45)
    )
    save_plot_as_html(
        fig16,
        "task6_cumulative_installs_area.html",
        "This stacked area chart shows cumulative installs over time for filtered T and P categories, with intensified area overlays on months where installs grew by more than 25% month over month."
    )

# Split plot_containers to handle the last plot properly
plot_containers_split = plot_containers.split('</div>')
if len(plot_containers_split) > 1:
    final_plot = plot_containers_split[-2] + '</div>'
else:
    final_plot = plot_containers  # Use plot_containers as default if splitting isn't sufficient

# HTML template for the dashboard
dashboard_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Google Play Store Reviews Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444;
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555;
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0, 0, 0, 0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container:hover .insights {{
            display: block;
        }}
    </style>
    <script>
        function openPlot(filename) {{
            window.open(filename, '_blank');
        }}
        function isWindowIST(startHour, endHour) {{
            const now = new Date();
            const utcTime = now.getTime() + now.getTimezoneOffset() * 60000;
            const istTime = new Date(utcTime + 330 * 60000);
            const minutes = istTime.getHours() * 60 + istTime.getMinutes();
            return minutes >= startHour * 60 && minutes <= endHour * 60;
        }}
        window.addEventListener('DOMContentLoaded', function() {{
            const task1Plot = document.getElementById('task1_rating_reviews_by_category.html');
            const task2Plot = document.getElementById('task2_global_installs_choropleth.html');
            const task3Plot = document.getElementById('task3_installs_revenue_free_paid.html');
            const task4Plot = document.getElementById('task4_installs_time_series.html');
            const task5Plot = document.getElementById('task5_size_rating_bubble.html');
            const task6Plot = document.getElementById('task6_cumulative_installs_area.html');
            if (task1Plot && !isWindowIST(15, 17)) {{
                task1Plot.style.display = 'none';
            }}
            if (task2Plot && !isWindowIST(18, 20)) {{
                task2Plot.style.display = 'none';
            }}
            if (task3Plot && !isWindowIST(13, 14)) {{
                task3Plot.style.display = 'none';
            }}
            if (task4Plot && !isWindowIST(18, 21)) {{
                task4Plot.style.display = 'none';
            }}
            if (task5Plot && !isWindowIST(17, 19)) {{
                task5Plot.style.display = 'none';
            }}
            if (task6Plot && !isWindowIST(16, 18)) {{
                task6Plot.style.display = 'none';
            }}
        }});
    </script>
</head>
<body>
    <div class="header">
        <img src="https://upload.wikimedia.org/wikipedia/commons/5/51/Google.png?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=original" alt="Google Logo">
        <h1>Google Play Store Reviews Analytics</h1>
        <img src="https://upload.wikimedia.org/wikipedia/commons/3/3d/Google_Play_2016_logo.png" alt="Google Play Store Logo">
    </div>
    <div class="container">
        {plots}
    </div>
</body>
</html>
"""

# Use these containers to fill in your dashboard HTML
final_html = dashboard_html.format(plots=plot_containers, plot_width=plot_width, plot_height=plot_height)

# Save the final dashboard to an HTML file
dashboard_path = os.path.join(html_files_path, "dashboard.html")
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

# Automatically open the generated HTML file in a web browser
webbrowser.open('file://' + os.path.realpath(dashboard_path))

True